# Quantum QPUF — Noise Mitigation with Mitiq

Local, noisy **density-matrix** simulation of the Quantum QPUF (PE-QPUF) circuit in **Cirq**, with error mitigation applied via **[Mitiq](https://mitiq.readthedocs.io)**.

The QPUF is a phase-estimation construction (see `submit_QPUF_ntarg.py`): a target register is prepared in a random state, then one or more QPE stages read the phase of a **Haar-random unitary** `U` acting on the target. We mitigate the **probability of the correct phase bitstring** — i.e. `Tr[ρ Π_b]` where `b` is the bitstring the *noiseless* QPUF outputs with highest probability.

## Scope of this notebook
- **Mitiq error mitigation only**, fully local and free.
- IonQ's native **debiasing** is a *hardware* feature (it can't run on a local simulator). It lives in a separate submission script, **`ionq_noise_mitig.py`** (1 target qubit; recommended max `N_PREC = 6` at the ≥2500-shot debiasing floor).

## ⚠️ Local-simulator qubit limits
The noisy executor uses `cirq.DensityMatrixSimulator`, which stores a `2ⁿ × 2ⁿ` matrix = `16·4ⁿ` bytes:

| qubits | density-matrix RAM |
|-------:|-------------------:|
| 10     | 16 MB              |
| 12     | 268 MB             |
| 13     | 1.1 GB             |
| 14     | 4.3 GB             |
| 15     | 17 GB              |

**Density-matrix noisy simulation is practical only up to ~13–14 qubits.** The real hardware QPUF (`N_TARG=1, N_PREC=15` → **31 qubits**) is far out of reach here, so we use two reduced instances:

1. **4-qubit single-stage** QPUF — `N_PREC=3, N_TARG=1` (fast, the clean demo).
2. **10-qubit two-stage** QPUF — `N_PREC=4, N_TARG=2` (full PE-QPUF construction, matches the hardware shape).

ZNE noise-scaling folds the circuit, increasing *depth* not *width*, so the table above still bounds what we can run.

> **Two-stage deferred-measurement note.** The hardware two-stage QPUF mid-circuit-measures the stage-1 precision register (`prec_a`) to collapse the target. Because `prec_a` is **never touched again**, the *deferred-measurement principle* lets us drop that mid-circuit measurement and measure everything terminally with **identical output statistics** — which keeps the circuit measurement-free so Mitiq's folding works cleanly.

In [ ]:
import time
import numpy as np
import cirq
import mitiq
from mitiq import zne, ddd, rem
import matplotlib.pyplot as plt

print("cirq   :", cirq.__version__)
print("mitiq  :", mitiq.__version__)
np.set_printoptions(precision=4, suppress=True)

## Section 0 — The QPUF circuit in Cirq

We rebuild the QPUF natively in Cirq (rather than converting the Qiskit version) — cleaner for arbitrary controlled unitaries + inverse QFT at these small sizes. The pieces mirror `submit_QPUF_ntarg.py`:

- **`haar_random_unitary`** — Mezzadri sampling (complex Ginibre → QR → diagonal phase fix).
- **`stable_matrix_power`** — `U^{2^k}` re-projected onto the nearest unitary via SVD.
- **`add_qpe_stage`** — `H` on the precision register, controlled-`U^{2^{n-1-k}}`, then inverse QFT.
- **`build_qpuf`** — target init (`RY+RZ` per qubit) + one or two QPE stages. **Measurement-free** (the executor reads the density matrix directly), so folding stays valid.

Qubit layout: precision block(s) first, target qubits last — so marginalizing the density-matrix diagonal over the trailing target axes gives the precision-register distribution.

In [ ]:
def haar_random_unitary(d, rng):
    """Haar-distributed d x d unitary (Mezzadri, arXiv:math-ph/0609050)."""
    Z = (rng.standard_normal((d, d)) + 1j * rng.standard_normal((d, d))) / np.sqrt(2.0)
    Q, R = np.linalg.qr(Z)
    Lambda = np.diag(R) / np.abs(np.diag(R))
    return Q * Lambda


def stable_matrix_power(U, power):
    """U**power, re-projected onto the nearest unitary via SVD (polar decomp)."""
    M = np.linalg.matrix_power(U, power)
    W, _, Vh = np.linalg.svd(M)
    return W @ Vh


def add_qpe_stage(circuit, prec, targ, U):
    """One QPE stage in place: H on prec, controlled-U^{2^{n-1-k}}, inverse QFT."""
    n = len(prec)
    circuit.append(cirq.H.on_each(*prec))
    for k in range(n):
        power = 2 ** (n - 1 - k)
        gate = cirq.MatrixGate(stable_matrix_power(U, power), name=f"U^{power}").controlled(1)
        circuit.append(gate.on(prec[k], *targ))     # [control, *targets]
    circuit.append(cirq.qft(*prec, inverse=True))


def build_qpuf(n_prec, n_targ, U, two_stage, init_seed=99):
    """
    Measurement-free QPUF circuit.

    Layout: [prec_a ... (prec_b ...)] then [targ ...].
    two_stage=True drops the hardware mid-circuit measurement on prec_a and
    relies on the deferred-measurement principle (prec_a is never reused), so
    terminal measurement of (prec_a, prec_b) reproduces the MCM statistics.
    Returns: (circuit, qubits, prec_indices, targ_indices).
    """
    n_prec_total = 2 * n_prec if two_stage else n_prec
    n = n_prec_total + n_targ
    qs = cirq.LineQubit.range(n)
    prec_all, targ = qs[:n_prec_total], qs[n_prec_total:]

    circuit = cirq.Circuit()
    init_rng = np.random.default_rng(init_seed)
    for q in targ:                                   # random target state
        circuit.append(cirq.ry(init_rng.uniform(0, np.pi)).on(q))
        circuit.append(cirq.rz(init_rng.uniform(0, 2 * np.pi)).on(q))

    if two_stage:
        add_qpe_stage(circuit, prec_all[:n_prec], targ, U)   # stage 1 (prec_a)
        add_qpe_stage(circuit, prec_all[n_prec:], targ, U)   # stage 2 (prec_b)
    else:
        add_qpe_stage(circuit, prec_all, targ, U)

    return circuit, qs, list(range(n_prec_total)), list(range(n_prec_total, n))


def measured_marginal(rho, n_qubits, targ_idx):
    """P over the precision register = diagonal of rho marginalized over targets."""
    probs = np.real(np.diagonal(rho)).reshape([2] * n_qubits)
    return probs.sum(axis=tuple(targ_idx)).reshape(-1)

### Build the two QPUF instances + define the observable

For each case we build the circuit, run a **noiseless** density-matrix simulation, and take the **argmax** of the precision-register marginal as the *correct phase bitstring* `b`. The mitigation target is `P(b)` — a single scalar Mitiq can extrapolate.

The `noise` per case is a per-moment depolarizing rate applied to **every** qubit (matching the Mitiq sample's `circuit.with_noise(cirq.depolarize(p))`).

In [ ]:
SIM = cirq.DensityMatrixSimulator()

# Per-case config. `zne_scales` kept smaller for the 10q case (folding a
# 10-qubit density matrix is the expensive step ~tens of seconds).
CASE_CONFIG = [
    dict(name="4q single-stage",  n_prec=3, n_targ=1, two_stage=False,
         noise=0.010, seed=10, zne_scales=[1, 3, 5]),
    dict(name="10q two-stage",    n_prec=4, n_targ=2, two_stage=True,
         noise=0.005, seed=10, zne_scales=[1, 3]),
]

CASES = {}
for cfg in CASE_CONFIG:
    rng = np.random.default_rng(cfg["seed"])
    U = haar_random_unitary(2 ** cfg["n_targ"], rng)
    circuit, qs, prec_idx, targ_idx = build_qpuf(
        cfg["n_prec"], cfg["n_targ"], U, cfg["two_stage"])

    rho_ideal = SIM.simulate(circuit, qubit_order=qs).final_density_matrix
    marg_ideal = measured_marginal(rho_ideal, len(qs), targ_idx)
    correct_idx = int(np.argmax(marg_ideal))
    correct_bits = format(correct_idx, f"0{len(prec_idx)}b")

    CASES[cfg["name"]] = dict(
        cfg=cfg, circuit=circuit, qs=qs, prec_idx=prec_idx, targ_idx=targ_idx,
        correct_idx=correct_idx, correct_bits=correct_bits,
        ideal=float(marg_ideal[correct_idx]),
    )
    print(f"{cfg['name']:18s}: {len(qs)} qubits, depth {len(circuit)}, "
          f"{len(prec_idx)} precision bits, noise p={cfg['noise']}")
    print(f"{'':18s}  correct bitstring b = '{correct_bits}',  ideal P(b) = "
          f"{CASES[cfg['name']]['ideal']:.4f}")

### The noisy executor

Mitiq calls an **executor**: circuit → expectation value. Ours injects per-moment depolarizing noise, simulates the density matrix, and returns `P(correct bitstring)`. When Mitiq folds the circuit for ZNE, the folded (deeper) copy accrues proportionally more noise — exactly the noise-scaling ZNE relies on.

In [ ]:
def make_executor(case, noise_level=None):
    """circuit -> P(correct bitstring) under per-moment depolarizing noise."""
    qs, targ_idx, correct_idx = case["qs"], case["targ_idx"], case["correct_idx"]
    n = len(qs)
    p = case["cfg"]["noise"] if noise_level is None else noise_level

    def execute(circuit):
        noisy = circuit.with_noise(cirq.depolarize(p=p)) if p > 0 else circuit
        rho = SIM.simulate(noisy, qubit_order=qs).final_density_matrix
        return float(measured_marginal(rho, n, targ_idx)[correct_idx])

    return execute


def rel_err(case, value):
    return abs((case["ideal"] - value) / case["ideal"])


# Baseline: ideal vs. unmitigated noisy P(b)
for name, case in CASES.items():
    case["executor"] = make_executor(case)
    case["noisy"] = case["executor"](case["circuit"])
    print(f"{name:18s}: ideal P(b)={case['ideal']:.4f}  "
          f"noisy P(b)={case['noisy']:.4f}  (rel. error {rel_err(case, case['noisy']):.3f})")

## EM tool 1 — Zero-Noise Extrapolation (ZNE)

Run the circuit at several **amplified** noise levels (via global folding `C → C C† C ...`, scale factors `1, 3, 5, …`) and extrapolate back to zero noise. We compare two extrapolators:

- **Richardson** — exact polynomial fit through all points.
- **Exponential** — fits `a + b·exp(−c·λ)`, often more robust for depolarizing-type decay.

> The 10q case folds a 10-qubit density matrix; with scales `[1,3]` expect ~10–20 s.

In [ ]:
for name, case in CASES.items():
    scales = case["cfg"]["zne_scales"]
    t0 = time.time()

    rich = zne.execute_with_zne(
        case["circuit"], case["executor"],
        factory=zne.inference.RichardsonFactory(scale_factors=scales),
        scale_noise=zne.scaling.fold_global,
    )
    case["zne_richardson"] = rich

    # ExpFactory fits 3 parameters -> needs >= 3 scale factors.
    if len(scales) >= 3:
        exp = zne.execute_with_zne(
            case["circuit"], case["executor"],
            factory=zne.inference.ExpFactory(scale_factors=scales, asymptote=None),
            scale_noise=zne.scaling.fold_global,
        )
        case["zne_exp"] = exp
        exp_str = f"  Exp={exp:.4f}(err {rel_err(case, exp):.3f})"
    else:
        case["zne_exp"] = None
        exp_str = "  Exp=n/a (needs >=3 scales)"

    print(f"{name:18s}: scales={scales}  noisy={case['noisy']:.4f}(err {rel_err(case, case['noisy']):.3f})"
          f"  Richardson={rich:.4f}(err {rel_err(case, rich):.3f}){exp_str}  [{time.time()-t0:.1f}s]")

## EM tool 2 — Digital Dynamical Decoupling (DDD)

DDD inserts decoupling pulse sequences (e.g. `XYXY`) into **idle windows** to average out coherent/dephasing errors during qubit idle time.

**Caveat for QPE/QPUF:** these circuits are gate-dense with little idle slack, so DDD often inserts few or no sequences and barely moves `P(b)` — and against a *purely depolarizing* noise model (no idle-time dephasing) there is little for it to cancel. We include it to show the API and to make the point honestly: **DDD pays off on circuits with real idle time + non-depolarizing dephasing**, not on these. Try the `yy` rule or a larger spacing to compare.

In [ ]:
for name, case in CASES.items():
    val = ddd.execute_with_ddd(case["circuit"], case["executor"], rule=ddd.rules.xyxy)
    case["ddd"] = val
    print(f"{name:18s}: noisy={case['noisy']:.4f}(err {rel_err(case, case['noisy']):.3f})  "
          f"DDD[xyxy]={val:.4f}(err {rel_err(case, val):.3f})")

## EM tool 3 — Readout-Error Mitigation (REM)

REM corrects **measurement (readout) error** by inverting a confusion matrix `A`, where `A[i,j] = P(measure i | prepared j)`. Our depolarizing executor models *gate* noise but not readout error, so here we **add an explicit readout bit-flip channel** (`p0 = p1 = 0.03`) on top of the gate-noisy distribution, then undo it with Mitiq's `generate_inverse_confusion_matrix`:

1. `p_gate` — precision-register distribution under gate noise only.
2. `p_readout = A · p_gate` — corrupted by symmetric readout bit-flips.
3. `p_rem = A⁻¹ · p_readout` (clipped, renormalized) — REM recovers `p_gate`.

So REM should remove the *readout* contribution exactly, leaving only the residual gate noise (which is ZNE/DDD territory).

In [ ]:
P0 = P1 = 0.03   # symmetric readout bit-flip probability per measured qubit

for name, case in CASES.items():
    nb = len(case["prec_idx"])
    ci = case["correct_idx"]

    # gate-noisy precision-register distribution
    noisy = case["circuit"].with_noise(cirq.depolarize(p=case["cfg"]["noise"]))
    rho = SIM.simulate(noisy, qubit_order=case["qs"]).final_density_matrix
    p_gate = measured_marginal(rho, len(case["qs"]), case["targ_idx"])

    A_inv = rem.generate_inverse_confusion_matrix(nb, p0=P0, p1=P1)
    A = np.linalg.inv(A_inv)                       # forward confusion matrix
    p_readout = A @ p_gate                         # add readout error
    p_rem = np.clip(A_inv @ p_readout, 0, None)    # REM correction
    p_rem /= p_rem.sum()

    case["rem_readout_noisy"] = float(p_readout[ci])
    case["rem_corrected"] = float(p_rem[ci])
    print(f"{name:18s}: ideal={case['ideal']:.4f}  gate+readout={p_readout[ci]:.4f}"
          f"(err {rel_err(case, p_readout[ci]):.3f})  "
          f"REM-corrected={p_rem[ci]:.4f}(err {rel_err(case, p_rem[ci]):.3f})")

## Summary — relative error in `P(correct bitstring)` by technique

Lower is better. ZNE should dominate (it targets the gate noise that dominates these circuits); REM only helps once readout error is present; DDD is near-flat against depolarizing noise on dense circuits.

In [ ]:
import os

# Methods compared on the SAME (gate-noise) baseline: unmitigated, ZNE, DDD.
gate_methods = ["noisy", "zne_richardson", "zne_exp", "ddd"]
gate_labels = ["Unmitigated", "ZNE (Rich.)", "ZNE (Exp.)", "DDD"]

fig, axes = plt.subplots(1, len(CASES), figsize=(6 * len(CASES), 4.2), squeeze=False)
for ax, (name, case) in zip(axes[0], CASES.items()):
    errs, labs = [], []
    for key, lab in zip(gate_methods, gate_labels):
        v = case.get(key)
        if v is None:
            continue
        errs.append(rel_err(case, v))
        labs.append(lab)
    bars = ax.bar(labs, errs, color=["#888", "#1f77b4", "#2ca02c", "#ff7f0e"][:len(labs)])
    ax.set_title(f"{name}\nideal P(b)={case['ideal']:.3f}, b='{case['correct_bits']}'")
    ax.set_ylabel("relative error in P(b)")
    ax.tick_params(axis="x", rotation=15)
    for bar, e in zip(bars, errs):
        ax.text(bar.get_x() + bar.get_width() / 2, e, f"{e:.3f}",
                ha="center", va="bottom", fontsize=9)
fig.suptitle("Mitiq error mitigation on the Quantum QPUF (gate-noise baseline)", y=1.02)
fig.tight_layout()

plots_dir = os.path.join(os.getcwd(), "plots")
os.makedirs(plots_dir, exist_ok=True)
out_png = os.path.join(plots_dir, "noise_mitigation_summary.png")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print("saved:", out_png)
plt.show()

# Text table including the REM (readout-error) comparison
print(f"\n{'case':18s} {'ideal':>7} {'noisy':>7} {'ZNE-R':>7} {'ZNE-E':>7} {'DDD':>7} "
      f"{'RO-noisy':>9} {'REM':>7}")
for name, case in CASES.items():
    def fmt(v): return f"{v:7.4f}" if v is not None else f"{'n/a':>7}"
    print(f"{name:18s} {case['ideal']:7.4f} {case['noisy']:7.4f} "
          f"{fmt(case['zne_richardson'])} {fmt(case['zne_exp'])} {fmt(case['ddd'])} "
          f"{case['rem_readout_noisy']:9.4f} {case['rem_corrected']:7.4f}")

## Notes & further tools

- **Scaling up.** Density-matrix noise simulation tops out near 13–14 qubits (see the table at the top). To push wider, switch to a statevector + Monte-Carlo trajectory executor (returns sampled bitstrings instead of `ρ`); Mitiq's ZNE/DDD/REM all accept executors that return a `MeasurementResult`.
- **More Mitiq techniques** to try on this circuit:
  - **PEC** (Probabilistic Error Cancellation) — `mitiq.pec`; needs quasi-probability representations of the noisy gates. Most powerful, most expensive.
  - **CDR / vnCDR** (Clifford Data Regression) — `mitiq.cdr`; trains on near-Clifford versions of the QPUF, then corrects. A good fit when an efficient classical near-Clifford simulation exists.
- **IonQ debiasing (hardware).** Run the companion script `ionq_noise_mitig.py` to submit the 1-target-qubit QPUF to IonQ with native debiasing (`device_parameters={"errorMitigation": Debias()}`, ≥2500 shots). Its built-in calculator recommends **max `N_PREC = 6`** before the per-shot gate budget pushes estimated circuit fidelity below 0.5 — edit `F2Q/F1Q` to your latest calibration.